# Mission 1 / 2 · 음성 파인튜닝 (Colab GPU)로컬 CPU 베이스라인은 MFCC/F0 통계 + 얕은 분류기. 이 노트북은 wav2vec2 를 파인튜닝한다.**중요**: 원본은 8 kHz 협대역이고 wav2vec2 는 16 kHz 광대역 사전학습이다. 단순 업샘플만으로도동작하지만 대역 불일치로 성능이 깎인다 (Sivaraman & Khoury, Odyssey'20). 업샘플은 GPU 쪽에서수행하고, 업로드는 8 kHz 원본으로 해 용량을 절반으로 줄인다.`python -m src.preprocess.pack_for_colab --what audio` 로 만든 npz 를 `MyDrive/dcc/` 에 올릴 것.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')DIR = '/content/drive/MyDrive/dcc'!pip -q install "transformers>=4.44" torchaudio scikit-learnimport torch, numpy as np, torchaudioprint('cuda', torch.cuda.is_available())

In [ ]:
MISSION = 'm2'      # 'm1' = 성별, 'm2' = 화자 역할def load(split):    d = np.load(f'{DIR}/{MISSION}_audio_{split}.npz')    a, off, y = d['audio'], d['offsets'], d['y']    segs = [a[off[i]:off[i+1]] for i in range(len(off)-1)]    extra = {k: d[k] for k in ('overlap','call') if k in d}    return segs, y.astype(np.int64), extratr_x, tr_y, _ = load('train')va_x, va_y, va_extra = load('val')print(f'train {len(tr_x):,}  val {len(va_x):,}  양성비율 {tr_y.mean():.3f}')

In [ ]:
from torch.utils.data import Dataset, DataLoaderfrom transformers import AutoFeatureExtractor, AutoModelForAudioClassificationMODEL = 'facebook/wav2vec2-base'SEC = 3.0                      # 고정 길이 창 (짧으면 pad, 길면 center-crop)SR_IN, SR_OUT = 8000, 16000resamp = torchaudio.transforms.Resample(SR_IN, SR_OUT)N = int(SEC * SR_OUT)fe = AutoFeatureExtractor.from_pretrained(MODEL)model = AutoModelForAudioClassification.from_pretrained(MODEL, num_labels=2).cuda()class ADS(Dataset):    def __init__(self, segs, y): self.s, self.y = segs, y    def __len__(self): return len(self.s)    def __getitem__(self, i):        w = torch.from_numpy(self.s[i].astype(np.float32) / 32768.0)        w = resamp(w)        if len(w) < N:            w = torch.nn.functional.pad(w, (0, N - len(w)))        elif len(w) > N:            o = (len(w) - N) // 2; w = w[o:o+N]        return w, self.y[i]def collate(b):    x = torch.stack([i[0] for i in b])    x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-7)    return x, torch.tensor([i[1] for i in b])tr_dl = DataLoader(ADS(tr_x, tr_y), batch_size=16, shuffle=True, collate_fn=collate, num_workers=2)va_dl = DataLoader(ADS(va_x, va_y), batch_size=32, collate_fn=collate, num_workers=2)

In [ ]:
from torch.optim import AdamWopt = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)scaler = torch.amp.GradScaler('cuda')@torch.no_grad()def evaluate():    model.eval(); P = []    for x, _ in va_dl:        with torch.amp.autocast('cuda'):            P.append(model(input_values=x.cuda()).logits.float().argmax(-1).cpu().numpy())    return np.concatenate(P)for ep in range(2):    model.train(); tot = 0    for i, (x, y) in enumerate(tr_dl):        opt.zero_grad(set_to_none=True)        with torch.amp.autocast('cuda'):            out = model(input_values=x.cuda(), labels=y.cuda())        scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update()        tot += out.loss.item()        if (i+1) % 200 == 0: print(f'ep{ep+1} {i+1}/{len(tr_dl)} loss {tot/(i+1):.4f}', flush=True)    p = evaluate()    acc = (p == va_y).mean()    print(f'== epoch {ep+1}  val acc {acc:.4f}')    if 'overlap' in va_extra:        ov = va_extra['overlap']        print(f'   비중첩 {(p[ov==0]==va_y[ov==0]).mean():.4f} / 중첩 {(p[ov==1]==va_y[ov==1]).mean():.4f}')

In [ ]:
import matplotlib.pyplot as plt, matplotlib!apt-get -qq install fonts-nanum > /dev/nullmatplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')matplotlib.rc('font', family='NanumGothic'); matplotlib.rc('axes', unicode_minus=False)p = evaluate()labels = ['여성 F','남성 M'] if MISSION=='m1' else ['119대원','신고자']cm = np.array([[((va_y==i)&(p==j)).sum() for j in (0,1)] for i in (0,1)], dtype=float)pct = cm/np.maximum(cm.sum(1,keepdims=True),1)fig, ax = plt.subplots(figsize=(4.2,3.8))ax.imshow(pct, cmap='Blues', vmin=0, vmax=1)ax.set_xticks([0,1], labels); ax.set_yticks([0,1], labels)ax.set_xlabel('예측'); ax.set_ylabel('정답')ax.set_title(f'{MISSION.upper()} wav2vec2  acc={(p==va_y).mean():.4f}')for i in range(2):    for j in range(2):        ax.text(j,i,f'{int(cm[i,j]):,}\n{pct[i,j]*100:.1f}%',ha='center',va='center',                color='white' if pct[i,j]>0.55 else '#16202B')plt.tight_layout(); plt.show()torch.save({'state_dict': model.state_dict(), 'model_name': MODEL, 'sec': SEC},           f'{DIR}/ckpt/{MISSION}.pt')